# AEGIS Phase 1 — Exploratory Data Analysis (EDA)
### Dataset: CDC Behavioral Risk Factor Surveillance System (BRFSS) 2024
**Objective**: Analyze 457,670 respondent health profiles, demographic factors, chronic condition prevalences, and the physical health target (`_PHYS14D`).

> **Notice**: Research prototype. Not intended for direct clinical diagnosis or medical underwriting.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyreadstat
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 30)


## 1. Load Raw BRFSS Sample & Inspect Candidate Variables


In [ ]:
raw_path = Path('../../data/raw/LLCP2024.XPT')
cols_to_load = [
    '_PHYS14D', 'PHYSHLTH', '_RFHLTH', 'POORHLTH', 'GENHLTH',
    '_AGE_G', 'SEXVAR', '_BMI5', '_SMOKER3', 'DIABETE4',
    'CVDINFR4', 'CVDCRHD4', 'CVDSTRK3', 'ASTHMA3', 'CHCCOPD3',
    'CHCKDNY2', 'HAVARTH4', 'EXERANY2', 'EDUCA', 'INCOME3',
    'PRIMINS2', 'PERSDOC3', 'MEDCOST1'
]

df, meta = pyreadstat.read_xport(str(raw_path), usecols=cols_to_load)
print(f"Total Loaded Rows: {len(df):,}, Columns: {df.shape[1]}")
df.head()


## 2. Target Variable Distribution (`_PHYS14D`)


In [ ]:
target_counts = df['_PHYS14D'].value_counts(dropna=False)
target_labels = {
    1.0: '0 days not good (58.5%)',
    2.0: '1-13 days not good (25.2%)',
    3.0: '14+ days not good [High Risk] (13.9%)',
    9.0: 'Refused / Missing (2.4%)'
}
print("Raw Target Frequency Table:")
for val, count in target_counts.items():
    lbl = target_labels.get(val, 'Unknown')
    print(f"Code {val}: {count:,} ({count / len(df):.2%}) -> {lbl}")


## 3. Prevalences of Chronic Conditions


In [ ]:
conditions = {
    'Diabetes (DIABETE4==1)': (df['DIABETE4'] == 1).mean() * 100,
    'Heart Attack (CVDINFR4==1)': (df['CVDINFR4'] == 1).mean() * 100,
    'Coronary Disease (CVDCRHD4==1)': (df['CVDCRHD4'] == 1).mean() * 100,
    'Stroke (CVDSTRK3==1)': (df['CVDSTRK3'] == 1).mean() * 100,
    'COPD / Emphysema (CHCCOPD3==1)': (df['CHCCOPD3'] == 1).mean() * 100,
    'Kidney Disease (CHCKDNY2==1)': (df['CHCKDNY2'] == 1).mean() * 100,
    'Arthritis (HAVARTH4==1)': (df['HAVARTH4'] == 1).mean() * 100,
    'Active Smoker (_SMOKER3 in [1,2])': df['_SMOKER3'].isin([1, 2]).mean() * 100,
}

cond_df = pd.Series(conditions).sort_values(ascending=True)
plt.figure(figsize=(9, 5))
cond_df.plot(kind='barh', color='#3b82f6')
plt.title('CDC BRFSS 2024 — Chronic Condition & Risk Factor Prevalences (%)', fontsize=12, fontweight='bold')
plt.xlabel('Prevalence (%)')
plt.tight_layout()
plt.show()
